# recon4d - full-size benchmark on GPU

Runs the complete benchmark of [recon4d](https://github.com/francoisbarge8/recon4d) on a
Kaggle notebook with two T4 GPUs.

**Before running**: in the notebook settings choose *Accelerator: GPU T4 x2* and turn
*Internet* on (needed to clone the repository and download pretrained weights).

| step | what | time |
|---|---|---|
| 1 | clone and install | 1 min |
| 2 | unit tests and a GPU smoke test | 4 min |
| 3 | train the in-domain depth network | 15 min |
| 4 | quick pass: 3 variants x 4 scenes | 30 min |
| 5 | all variants | 2-3 h |
| 6 | optional: cross-check against gsplat, real video | 10 min |
| 7 | zip the results | - |

Every step is resumable: re-running a cell skips what is already done.

In [ ]:
REPO_URL = "https://github.com/francoisbarge8/recon4d.git"  # edit if the repository lives elsewhere

import os
import subprocess

WORK = "/kaggle/working"
token = None
try:  # only needed for a private repository: add a secret named GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
repo = f"{WORK}/recon4d"
if os.path.exists(repo):
    origin = subprocess.run(
        ["git", "-C", repo, "remote", "get-url", "origin"], capture_output=True, text=True
    ).stdout.strip()
    # A clone of another URL, or of an empty repository, is replaced by a fresh one.
    if origin.split("@")[-1] != url.split("@")[-1] or not os.path.exists(f"{repo}/pyproject.toml"):
        subprocess.run(["rm", "-rf", repo], check=True)
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, repo], check=True)
assert os.path.exists(f"{repo}/pyproject.toml"), f"{REPO_URL} looks empty: check REPO_URL"
os.chdir(repo)
!pip install -q -e ".[lpips,depth]"
!pip install -q pycolmap || echo "pycolmap is unavailable: the colmap variant will fail and be skipped"
!git log --oneline | head -3
!nvidia-smi -L

## 2. Tests

The unit tests run on the CPU; the smoke test runs the whole pipeline on the GPU on tiny sequences.

In [ ]:
!python -m pytest -q -x tests 2>&1 | tail -3

In [ ]:
!python -m recon4d.cli benchmark --profile smoke --out {WORK}/smoke --device cuda --scenes rolling,still --variants full,oracle-all,static-only 2>&1 | grep -v Warning | tail -5
print(open(f"{WORK}/smoke/results.md").read())

## 3. Depth network

A compact U-Net trained on random scenes drawn from the benchmark's scene grammar (never on
the benchmark scenes themselves). The rendered training set is cached.

In [ ]:
!python scripts/train_depth.py --out assets/checkpoints/tiny_depth.pth --workers 4 --cache {WORK}/depth_cache 2>&1 | grep -v Warning | tail -45

## 4. Quick pass

The default pipeline, the static-only baseline and the all-oracle upper bound on the four
scenes, one worker per GPU.

In [ ]:
BASE = "depth=learned depth_checkpoint=assets/checkpoints/tiny_depth.pth flow=raft train.raster.chunk_elements=40000000"
OUT = f"{WORK}/results"
!python scripts/run_benchmark_multi_gpu.py --out {OUT} --profile gpu --lpips alex --variants full,static-only,oracle-all {BASE}
!tail -3 {OUT}/worker0.log
print(open(f"{OUT}/results.md").read())

## 5. All variants

Oracle analysis, ablations and back-end swaps (Depth Anything V2, CoTracker3, COLMAP poses).
A run that fails is skipped: its traceback is written to `error.txt` in its folder.

In [ ]:
VARIANTS = "full,static-only,oracle-depth,oracle-tracks,oracle-poses,oracle-all,no-depth-loss,no-track-loss,no-rigidity,no-depth-correction,depth-prior-ba,depth-anything,cotracker,colmap"
!python scripts/run_benchmark_multi_gpu.py --out {OUT} --profile gpu --lpips alex --variants {VARIANTS} {BASE}
print(open(f"{OUT}/results.md").read())

In [ ]:
from IPython.display import Image, display

for scene in ("rolling", "sliding", "squash"):
    path = f"{OUT}/{scene}/full/reconstruction.gif"
    if os.path.exists(path):
        print(scene)
        display(Image(filename=path))

## 6. Optional

Cross-check of the pure-PyTorch rasterizer against gsplat's CUDA rasterizer.

In [ ]:
!pip install -q gsplat
!python scripts/compare_gsplat.py --out {OUT}/gsplat_check.json

A real video: set `VIDEO` to a file added to the notebook (*Add Input* or *Upload*). The
front-end then uses Depth Anything V2, KLT + RAFT and structure-from-motion with an unknown
focal length.

In [ ]:
VIDEO = ""  # e.g. "/kaggle/input/my-clip/clip.mp4"
if VIDEO:
    !python -m recon4d.cli video "{VIDEO}" --out {OUT}/video --max-frames 60 --max-side 384 depth=depth-anything flow=raft device=cuda train.iterations=7000 train.densify.stop=5000 train.densify.max_gaussians=120000 init.n_static=40000 train.raster.chunk_elements=40000000
    display(Image(filename=f"{OUT}/video/reconstruction.gif"))

## 7. Package the results

Download `results.zip` from the *Output* panel and unzip it at the root of the repository.

In [ ]:
!cd {WORK} && rm -f results.zip && zip -qr results.zip results smoke recon4d/assets/checkpoints -x "*/cache/*" && ls -lh results.zip